# 加载所有数据

In [7]:
import numpy as np

files = ['test_2_ssvep', 'test_8_ssvep']
freqs = [15, 12, 10]  # 假设label 0/1/2/3对应这4类

Xs = []
ys = []

for txt in files:
    x = np.load(f'data/eeg_trials_{txt}.npy', allow_pickle=True)
    y = np.load(f'data/eeg_labels_{txt}.npy')

    x = np.asarray(x)
    if x.dtype == object:
        x = np.stack(x)

    print(txt, x.shape, y.shape, np.bincount(y.astype(int)))

    Xs.append(x)
    ys.append(y.astype(int))

# 统一时间长度
min_T = min(x.shape[2] for x in Xs)
Xs = [x[:, :, :min_T] for x in Xs]

X = np.concatenate(Xs, axis=0)
y = np.concatenate(ys, axis=0)

print("X:", X.shape)
print("y:", y.shape)
print("count:", np.bincount(y))

test_2_ssvep (19, 8, 374) (19,) [8 4 7]
test_8_ssvep (67, 8, 373) (67,) [23 24 20]
X: (86, 8, 373)
y: (86,)
count: [31 28 27]


# 训练

In [8]:
def pearson_corr(a, b):
    a = a - a.mean()
    b = b - b.mean()
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)


def train_trca_one_class(Xc):
    """
    Xc: [n_trials, channels, time]
    """
    Xc = Xc - Xc.mean(axis=2, keepdims=True)

    n_trials, n_ch, n_time = Xc.shape

    S = np.zeros((n_ch, n_ch))
    Q = np.zeros((n_ch, n_ch))

    for i in range(n_trials):
        Xi = Xc[i]
        Q += Xi @ Xi.T

        for j in range(i + 1, n_trials):
            Xj = Xc[j]
            S += Xi @ Xj.T + Xj @ Xi.T

    reg = 1e-6 * np.eye(n_ch)
    eig_mat = np.linalg.pinv(Q + reg) @ S

    eigvals, eigvecs = np.linalg.eig(eig_mat)
    w = np.real(eigvecs[:, np.argmax(np.real(eigvals))])

    w = w / (np.linalg.norm(w) + 1e-12)

    template = Xc.mean(axis=0)

    return w, template


def train_trca(X_train, y_train):
    model = {}

    for cls in np.unique(y_train):
        Xc = X_train[y_train == cls]
        w, template = train_trca_one_class(Xc)
        model[cls] = {
            "w": w,
            "template": template
        }

    return model


def predict_trca_one(x, model):
    """
    x: [channels, time]
    """
    x = x - x.mean(axis=1, keepdims=True)

    scores = {}

    for cls, item in model.items():
        w = item["w"]
        template = item["template"]

        test_1d = w @ x
        temp_1d = w @ template

        scores[cls] = pearson_corr(test_1d, temp_1d)

    return max(scores, key=scores.get), scores


def predict_trca(X_test, model):
    preds = []

    for x in X_test:
        pred, _ = predict_trca_one(x, model)
        preds.append(pred)

    return np.array(preds)

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.5,
    random_state=42,
    stratify=y
)

model = train_trca(X_train, y_train)

y_pred = predict_trca(X_test, model)

acc = np.mean(y_pred == y_test)

print("TRCA acc:", acc)

for i, f in enumerate(freqs):
    cls_acc = np.mean(y_pred[y_test == i] == y_test[y_test == i])
    print(f"{f} Hz acc:", cls_acc)

pred_freqs = [freqs[i] for i in y_pred]
print(pred_freqs)

TRCA acc: 1.0
15 Hz acc: 1.0
12 Hz acc: 1.0
10 Hz acc: 1.0
[15, 12, 10, 12, 12, 12, 15, 15, 15, 10, 10, 12, 15, 10, 15, 12, 10, 15, 12, 10, 12, 15, 15, 12, 15, 15, 15, 10, 12, 12, 15, 10, 15, 15, 10, 15, 10, 12, 12, 10, 10, 10, 12]
